In [ ]:
# =========================
# 1) Install + Imports
# =========================
!pip -q install pymongo pandas numpy matplotlib scipy seaborn

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pymongo import MongoClient
from scipy.stats import pearsonr, spearmanr

sns.set_context("talk")


# =========================
# 2) Mongo Config (EDIT ME)
# =========================
MONGO_URI = os.environ.get(
    "MONGO_URI",
    "xxxxx"
)
DB_NAME = os.environ.get("DB_NAME", "cosmo_project")

KP_COLLECTION = os.environ.get("KP_COLLECTION", "kp_data")
SENSOR_COLLECTION = os.environ.get("SENSOR_COLLECTION", "sensor_data_latest")

client = MongoClient(MONGO_URI)
db = client[DB_NAME]


# =========================
# 3) Helper: load Mongo -> DataFrame
# =========================
def load_collection_as_df(collection_name: str, query=None, projection=None, limit=None) -> pd.DataFrame:
    query = query or {}
    cursor = db[collection_name].find(query, projection)
    if limit:
        cursor = cursor.limit(limit)
    docs = list(cursor)
    if not docs:
        raise ValueError(f"No documents found in collection: {collection_name}")
    return pd.json_normalize(docs)


# =========================
# 4) Load data
# =========================
kp_df_raw = load_collection_as_df(KP_COLLECTION)
sensor_df_raw = load_collection_as_df(SENSOR_COLLECTION)

print("KP columns:", kp_df_raw.columns[:20])
print("SENSOR columns:", sensor_df_raw.columns[:20])

kp_df_raw.head(), sensor_df_raw.head()


# =========================
# 5) Clean + Normalize timestamps
# =========================
def to_datetime_any(x):
    # handles:
    # - pandas Timestamp / datetime
    # - ISO strings
    # - dict like {"$date": "..."}
    if isinstance(x, dict) and "$date" in x:
        return pd.to_datetime(x["$date"], utc=True, errors="coerce")
    return pd.to_datetime(x, utc=True, errors="coerce")

# ---- KP
kp_df = kp_df_raw.copy()
kp_df["kp_time"] = kp_df.get("source_time_utc", kp_df.get("source_time_raw", None)).apply(to_datetime_any)
kp_df["kp_value"] = pd.to_numeric(kp_df["value"], errors="coerce")

kp_df = kp_df.dropna(subset=["kp_time", "kp_value"]).sort_values("kp_time")
kp_df = kp_df[["kp_time", "kp_value", "status", "index_name"]].reset_index(drop=True)

# ---- SENSOR
sensor_df = sensor_df_raw.copy()
sensor_df["sensor_time"] = sensor_df.get("sensor_time_utc", sensor_df.get("received_at_utc", None)).apply(to_datetime_any)

sensor_df["hr"] = pd.to_numeric(sensor_df.get("metrics.hr"), errors="coerce")
sensor_df["rmssd"] = pd.to_numeric(sensor_df.get("metrics.rmssd"), errors="coerce")

sensor_df = sensor_df.dropna(subset=["sensor_time"]).sort_values("sensor_time")

# rmssd=-1 invalid -> set to NaN
sensor_df.loc[sensor_df["rmssd"] <= 0, "rmssd"] = np.nan

sensor_df = sensor_df.reset_index(drop=True)

kp_df.head(), sensor_df.head()


# =========================
# 6) Choose overlap window (recommended)
# =========================
start = max(kp_df["kp_time"].min(), sensor_df["sensor_time"].min())
end = min(kp_df["kp_time"].max(), sensor_df["sensor_time"].max())

kp_df_w = kp_df[(kp_df["kp_time"] >= start) & (kp_df["kp_time"] <= end)].copy()
sensor_df_w = sensor_df[(sensor_df["sensor_time"] >= start) & (sensor_df["sensor_time"] <= end)].copy()

print("Overlap window:", start, "to", end)
print("KP points:", len(kp_df_w), "Sensor points:", len(sensor_df_w))


# =========================
# 7) Resample sensor HRV to match Kp cadence (REPLACED)
# =========================
RESAMPLE = "3H"

# Helper: compute RMSSD from IBI list (ms)
def rmssd_from_ibi_list(ibi_list_ms):
    """
    ibi_list_ms: list of IBI values in milliseconds
    RMSSD = sqrt(mean(diff(IBI)^2))
    """
    if not isinstance(ibi_list_ms, list) or len(ibi_list_ms) < 2:
        return np.nan
    arr = np.array(ibi_list_ms, dtype=float)
    diffs = np.diff(arr)
    return float(np.sqrt(np.mean(diffs**2)))

sensor_ts = sensor_df_w.copy().set_index("sensor_time").sort_index()

# If your Mongo docs store IBI as a list under column name "ibi", compute rmssd from it.
# If not present, it'll just be NaN and we fall back to metrics.rmssd.
if "ibi" in sensor_ts.columns:
    sensor_ts["rmssd_from_ibi"] = sensor_ts["ibi"].apply(rmssd_from_ibi_list)
else:
    sensor_ts["rmssd_from_ibi"] = np.nan

# Prefer measured rmssd, else computed rmssd from ibi
sensor_ts["rmssd_final"] = sensor_ts["rmssd"].combine_first(sensor_ts["rmssd_from_ibi"])

# Resample to match cadence
sensor_resampled = (
    sensor_ts.resample(RESAMPLE)
    .agg({
        "rmssd_final": "mean",
        "hr": "mean"
    })
    .rename(columns={"rmssd_final": "rmssd_mean", "hr": "hr_mean"})
    .reset_index()
    .rename(columns={"sensor_time": "time"})
)

# KP on same grid
kp_resampled = (
    kp_df_w.set_index("kp_time").sort_index()
    .resample(RESAMPLE)
    .agg({"kp_value": "mean"})
    .reset_index()
    .rename(columns={"kp_time": "time"})
)

# Robust alignment: match sensor buckets to Kp buckets (backward within tolerance)
merged = pd.merge_asof(
    sensor_resampled.sort_values("time"),
    kp_resampled.sort_values("time"),
    on="time",
    direction="backward",
    tolerance=pd.Timedelta(RESAMPLE)
)

merged = merged.dropna(subset=["kp_value", "rmssd_mean"]).sort_values("time").reset_index(drop=True)

print("Merged rows:", len(merged))
merged.head(), merged.describe()


# =========================
# 8) Plot: Time series (dual axis) - HR vs Kp
# =========================
plt.figure(figsize=(14,5))
ax1 = plt.gca()
ax2 = ax1.twinx()

ax1.plot(merged["time"], merged["hr_mean"], marker="o", linewidth=1)
ax2.plot(merged["time"], merged["kp_value"], marker="s", linewidth=1)

ax1.set_xlabel("Time (UTC)")
ax1.set_ylabel("Heart Rate mean (bpm)")
ax2.set_ylabel("Kp")

plt.title(f"Heart Rate (mean) vs Kp over time (resample={RESAMPLE})")
plt.tight_layout()
plt.show()


# =========================
# 9) Correlation stats + Scatter
# =========================
pear_r, pear_p = pearsonr(merged["rmssd_mean"], merged["kp_value"])
spear_r, spear_p = spearmanr(merged["rmssd_mean"], merged["kp_value"])

print(f"Pearson r = {pear_r:.3f}, p = {pear_p:.3g}")
print(f"Spearman rho = {spear_r:.3f}, p = {spear_p:.3g}")

plt.figure(figsize=(7,6))
sns.regplot(data=merged, x="kp_value", y="rmssd_mean", scatter_kws={"alpha":0.7})
plt.title("Scatter: Kp vs HRV (RMSSD mean)")
plt.xlabel("Kp")
plt.ylabel("RMSSD mean")
plt.tight_layout()
plt.show()


# =========================
# 10) Heatmap (optional)
# =========================
corr_df = merged[["kp_value", "rmssd_mean", "hr_mean"]].corr(method="spearman")
plt.figure(figsize=(6,4))
sns.heatmap(corr_df, annot=True, fmt=".2f")
plt.title("Spearman correlation (resampled)")
plt.tight_layout()
plt.show()

